In [3]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import numpy as np

# Show every single row
pd.set_option('display.max_rows', None)

# Show every single column
pd.set_option('display.max_columns', None)

# Show full text inside cells without cutting it off
pd.set_option('display.max_colwidth', None)

## Part 1: Conceptual Check

##### **Question 1:** You merge 6,840 sales rows with a 5-row outlet lookup table using `how="left"`, and the result has **7,200** rows. Nothing errored. What happened, and which one argument would have turned this into a loud failure instead of a silent one?
> **Answer:** 
> The lookup table has duplicate `outlet_id` values. Add `validate="m:1"` to raise `MergeError`.

In [4]:
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])
sales.info()
display(sales.outlet_id.dropna().unique())
# create simple outlet table with duplicate outlet_id (OUT-05) and outlet_name
outlets = pd.DataFrame({
    "outlet_id": ["OUT-01", "OUT-02", "OUT-03", "OUT-04", "OUT-05", "OUT-05"],
    "outlet_name": ["Raffles Place", "Tampines Mall", "Marina Bay", "Holland Village", "Kiosk", "Food Truck"]
})
outlets 
# If there are multiple matches in the right table for a single row in the left table, 
# it will create multiple rows in the result for that single row.
sales_outlets = sales.merge(outlets, on="outlet_id", how="left")
sales_outlets.info()

# Find duplicates in the outlets table
outlets[outlets.duplicated(subset="outlet_id", keep=False)]

# Flag the merge as valid or invalid using the `validate` argument.
sales_outlets_valid = sales.merge(outlets, on="outlet_id", how="left", validate="m:1")
sales_outlets_valid.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6840 entries, 0 to 6839
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         6840 non-null   datetime64[ns]
 1   outlet_id    6840 non-null   object        
 2   daypart      6840 non-null   object        
 3   tickets      6840 non-null   int64         
 4   items        6840 non-null   int64         
 5   revenue_sgd  6840 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 320.8+ KB


array(['OUT-01', 'OUT-02', 'OUT-03', 'OUT-04', 'OUT-05'], dtype=object)

,outlet_id,outlet_name
0,OUT-01,Raffles Place
1,OUT-02,Tampines Mall
2,OUT-03,Marina Bay
3,OUT-04,Holland Village
4,OUT-05,Kiosk
5,OUT-05,Food Truck


<class 'pandas.core.frame.DataFrame'>
Int64Index: 7116 entries, 0 to 7115
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         7116 non-null   datetime64[ns]
 1   outlet_id    7116 non-null   object        
 2   daypart      7116 non-null   object        
 3   tickets      7116 non-null   int64         
 4   items        7116 non-null   int64         
 5   revenue_sgd  7116 non-null   float64       
 6   outlet_name  7116 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 444.8+ KB


,outlet_id,outlet_name
4,OUT-05,Kiosk
5,OUT-05,Food Truck


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

##### **Question 2:** `df.resample("M").sum()` and `df.resample("M").mean()` on the same daily revenue give you two different pictures of February. Which one would you show a manager who asked "how did February go?", and what would you say alongside it?
> **Answer:** Show the **mean** (average trading day); but also mention the total for a complete picture. February has fewer days and total will be less. Weekends and public holidays also affects total.

In [21]:
df = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"], dayfirst=True, index_col="date")
df.resample("M").sum(numeric_only=True).round(0)
df.resample("M").mean(numeric_only=True).round(0)


,tickets,items,revenue_sgd
date,,,
2024-01-31,21517,31161,177818.0
2024-02-29,20277,29360,167402.0
2024-03-31,21946,31945,181769.0
2024-04-30,21874,31805,181174.0
2024-05-31,22409,32588,186426.0
2024-06-30,21285,30886,177374.0
2024-07-31,22887,33129,189085.0
2024-08-31,22427,32737,185997.0
2024-09-30,21222,30813,175610.0


,tickets,items,revenue_sgd
date,,,
2024-01-31,58.0,84.0,478.0
2024-02-29,58.0,84.0,481.0
2024-03-31,59.0,86.0,489.0
2024-04-30,61.0,88.0,503.0
2024-05-31,60.0,88.0,501.0
2024-06-30,59.0,86.0,493.0
2024-07-31,62.0,89.0,508.0
2024-08-31,60.0,88.0,500.0
2024-09-30,59.0,86.0,488.0


##### **Question 3:** You have monthly revenue with one column per month (wide), and monthly targets with a `month` column (long). Write the one line that makes them joinable, and say which of the two you would reshape.
> **Answer:** 
> `targets = targets_wide.melt(id_vars="outlet_id", var_name="month", value_name="target_sgd")`  
Reshape the **wide** one. Long format is the joinable format: a key has to exist as a column, and
in the wide layout "month" only exists as a set of headers.   
**Melt to compute, pivot to present.**

In [22]:
targets_wide = pd.read_csv("../data/targets_wide.csv")
targets_wide.head()

# 👉 `melt` unpivots. `id_vars` are the columns to KEEP as they are; everything else gets folded
#    down into two new columns: one holding the old header, one holding the value.
targets = targets_wide.melt(
    id_vars="outlet_id",        # keep this as a column
    var_name="month",           # the old column headers land here
    value_name="target_sgd",    # the numbers land here
)

print(f"{targets_wide.shape} wide  ->  {targets.shape} long")
targets.head()

,outlet_id,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06
0,OUT-01,51800.0,52000.0,52100.0,52300.0,52500.0,52700.0,52800.0,53000.0,53200.0,53300.0,53500.0,53700.0,53900.0,54000.0,54200.0,54400.0,54600.0,54800.0
1,OUT-02,45900.0,46000.0,46200.0,46300.0,46500.0,46600.0,46800.0,47000.0,47100.0,47300.0,47400.0,47600.0,47700.0,47900.0,48000.0,48200.0,48400.0,48500.0
2,OUT-03,46000.0,46100.0,46300.0,46400.0,46600.0,46700.0,46900.0,47000.0,47200.0,47300.0,47500.0,47600.0,47800.0,47900.0,48100.0,48300.0,48400.0,48600.0
3,OUT-04,32000.0,32100.0,32200.0,32300.0,32400.0,32500.0,32700.0,32800.0,32900.0,33000.0,33100.0,33200.0,33300.0,33400.0,33500.0,33600.0,33700.0,33800.0


(4, 19) wide  ->  (72, 3) long


,outlet_id,month,target_sgd
0,OUT-01,2024-01,51800.0
1,OUT-02,2024-01,45900.0
2,OUT-03,2024-01,46000.0
3,OUT-04,2024-01,32000.0
4,OUT-01,2024-02,52000.0


##### **Question 4:** `pd.crosstab(df["a"], df["b"])` and `df.pivot_table(index="a", columns="b", values="c", aggfunc="count")` can produce the same table. When would you deliberately reach for `crosstab` instead?
> **Answer:**   
> `crosstab`:  
> — works directly with lists and arrays  
> — built it `normalize` functions  
> — built-in `margin` function  
> — great for counting number of occurrences, frequencies  
> `pivot_table`:   
> — needs a DataFrame   
> — great for aggregation, especially with a `values` column

In [23]:
region = ["North", "South", "North", "South", "North"]
product = ["A", "B", "A", "A", "B"]
revenue = [100, np.nan, 300, np.nan, 500]

# crosstab — works directly on lists/arrays
pd.crosstab(region, product)
pd.crosstab(region, product, margins=True)  # add row and column totals
pd.crosstab(region, product, margins=True, margins_name="Total")  # change the name of the totals row/column
pd.crosstab(region, product, normalize=True)  # normalize by total
pd.crosstab(region, product, normalize="index")  # normalize by row
pd.crosstab(region, product, normalize="columns")  # normalize by column

# pivot_table — you'd have to build a DataFrame first
df = pd.DataFrame({"region": region, "product": product, "revenue": revenue})
pd.crosstab(df["region"], df["product"], normalize="index")  # normalize by row
pd.crosstab(df["region"], df["product"], normalize="columns")  # normalize by column

df.pivot_table(index="region", columns="product", aggfunc="size")
df.pivot_table(index="region", columns="product", aggfunc="count")    # count of non-NaN values
df.pivot_table(index="region", columns="product", values="revenue", aggfunc="sum")      # sum of values
df.pivot_table(index="region", columns="product", values="revenue", aggfunc="mean")      # mean of values
df.pivot_table(index="region", columns="product", values="revenue", aggfunc="std")      # standard deviation of values

col_0,A,B
row_0,,
North,2,1
South,1,1


col_0,A,B,All
row_0,,,
North,2,1,3
South,1,1,2
All,3,2,5


col_0,A,B,Total
row_0,,,
North,2,1,3
South,1,1,2
Total,3,2,5


col_0,A,B
row_0,,
North,0.4,0.2
South,0.2,0.2


col_0,A,B
row_0,,
North,0.666667,0.333333
South,0.500000,0.500000


...

product,A,B
region,,
North,2,1
South,1,1


revenue   
product       A  B
region            
North         2  1
South         0  0

product,A,B
region,,
North,400.0,500.0
South,0.0,0.0


product,A,B
region,,
North,200.0,500.0


product,A
region,
North,141.421356


##### **Question 5:** Two outlets have monthly revenues correlated at **−0.72**. Your colleague concludes that one is stealing customers from the other. Give two other explanations that fit the same number equally well, and say what you would check first.
> **Answer:** Correlation measures co-movement, not causation.  
> Some plausible explanations:  
(a) **confounders**   
— one is declining and the other growing over the same period for unrelated reasons  
(b) **other common causes**  
— shared staffing, shared budget, shared supplies, etc...  
(c) **Mean-reverting noise on a small sample**  
— with only 18 monthly values, a strong correlation is not hard to get by chance.   
What to check first:  
Plot both series and look at *when* each one moved. If one changed on a specific date and the other drifted throughout, cannibalisation is very unlikely. 
To confirm the "stealing" hypothesis, customer IDs would have to be captured and measure the actual "switched outlets" transactions.  

## Part 2: Practical Challenge: The Q3 Review Pack
Your Lesson 1.9 analysis landed well. The owner has now asked for a short pack ahead of the Q3
review, with four specific questions. Same data, new questions.

In [24]:
sales = pd.read_csv("../data/daily_sales.csv", parse_dates=["date"])
outlets = pd.read_csv("../data/outlets.csv", parse_dates=["opened_date"])
roster = pd.read_csv("../data/roster.csv", parse_dates=["week_start"])
targets_wide = pd.read_csv("../data/targets_wide.csv")

### Challenge 1: "Is Tampines Mall as steady as it looks?" (Time)
Tampines Mall (`OUT-02`) has been flat all year, which everyone has read as "stable".

In [25]:
# 1. Build a **weekly** revenue series for `OUT-02` (weeks starting Monday).
t = sales[sales.outlet_id == "OUT-02"].resample("W-MON", on="date", label="left").sum(numeric_only=True)
t.head()
t.tail()
# first date
sales[sales.outlet_id == "OUT-02"]["date"].min()
# last date
sales[sales.outlet_id == "OUT-02"]["date"].max()

,tickets,items,revenue_sgd
date,,,
2023-12-25,98,150,863.85
2024-01-01,1246,1802,10807.66
2024-01-08,1221,1796,10661.09
2024-01-15,1230,1759,10910.79
2024-01-22,1228,1818,10735.09


,tickets,items,revenue_sgd
date,,,
2025-05-26,1208,1737,11103.59
2025-06-02,1178,1716,10280.85
2025-06-09,1202,1710,10522.51
2025-06-16,1237,1815,10698.19
2025-06-23,1141,1655,10049.02


Timestamp('2024-01-01 00:00:00')

Timestamp('2025-06-30 00:00:00')

In [26]:
# 2. Add a **4-week rolling mean** column beside it.
# Discard first row because data is partial.
t = t.iloc[1:]
t["revenue_4wk_mean"] = t["revenue_sgd"].rolling(window=4).mean()
t.head()
t.tail()

,tickets,items,revenue_sgd,revenue_4wk_mean
date,,,,
2024-01-01,1246,1802,10807.66,NaN
2024-01-08,1221,1796,10661.09,NaN
2024-01-15,1230,1759,10910.79,NaN
2024-01-22,1228,1818,10735.09,10778.6575
2024-01-29,1266,1833,10913.29,10805.0650


,tickets,items,revenue_sgd,revenue_4wk_mean
date,,,,
2025-05-26,1208,1737,11103.59,11368.1750
2025-06-02,1178,1716,10280.85,11137.1325
2025-06-09,1202,1710,10522.51,10868.1300
2025-06-16,1237,1815,10698.19,10651.2850
2025-06-23,1141,1655,10049.02,10387.6425


In [27]:
# 3. Report its best and worst weeks by revenue.
best_week = t["revenue_sgd"].idxmax()
worst_week = t["revenue_sgd"].idxmin()
print(f"Best week: {best_week.date()}, Revenue: {t['revenue_sgd'].max()}")
print(f"Worst week: {worst_week.date()}, Revenue: {t['revenue_sgd'].min()}")
print(f"Spread: {t['revenue_sgd'].max() - t['revenue_sgd'].min():.2f}")
print(f"Standard Deviation: {t['revenue_sgd'].std():.2f}")
print(f"Coefficient of Variation: {t['revenue_sgd'].std() / t['revenue_sgd'].mean():.2f}")

Best week: 2024-08-12, Revenue: 11899.17
Worst week: 2024-04-29, Revenue: 9624.94
Spread: 2274.23
Standard Deviation: 498.23
Coefficient of Variation: 0.05


4. In one sentence: is "stable" the right word, or is it "noisy around a flat average"? Which
   column did you use to decide, and why?
> **Answer:** A CV of 0.05 is considered generally flat, using the column `revenue_sgd`.

### Challenge 2:  "Are we paying for staff we do not need?" (Joins)

In [28]:
# 1. Build weekly revenue per outlet (one row per outlet per week, weeks starting Monday).
roster.head()
roster.info()
# rename the date column to week_start to match the roster table
w = sales.groupby("outlet_id").resample("W-MON", on="date", label="left").sum(numeric_only=True).reset_index().rename(columns={"date": "week_start"})   
w.head()
w.info()

,outlet_id,week_start,staff_hours,headcount
0,OUT-01,2024-01-01,473.0,14
1,OUT-01,2024-01-08,421.6,12
2,OUT-01,2024-01-15,459.7,14
3,OUT-01,2024-01-22,446.1,13
4,OUT-01,2024-01-29,441.9,13


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   outlet_id    325 non-null    object        
 1   week_start   325 non-null    datetime64[ns]
 2   staff_hours  325 non-null    float64       
 3   headcount    325 non-null    int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 10.3+ KB


,outlet_id,week_start,tickets,items,revenue_sgd
0,OUT-01,2023-12-25,192,271,1464.97
1,OUT-01,2024-01-01,1486,2145,11785.71
2,OUT-01,2024-01-08,1607,2293,12761.65
3,OUT-01,2024-01-15,1469,2104,11830.20
4,OUT-01,2024-01-22,1556,2214,12165.29


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 330 entries, 0 to 329
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   outlet_id    330 non-null    object        
 1   week_start   330 non-null    datetime64[ns]
 2   tickets      330 non-null    int64         
 3   items        330 non-null    int64         
 4   revenue_sgd  330 non-null    float64       
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 13.0+ KB


In [29]:
# 2. Merge it with `roster` on **both** keys, and prove the merge did not lose or duplicate rows.
clean = w.merge(roster, on=["outlet_id", "week_start"], how="inner", validate="1:1")
clean.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 325 entries, 0 to 324
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   outlet_id    325 non-null    object        
 1   week_start   325 non-null    datetime64[ns]
 2   tickets      325 non-null    int64         
 3   items        325 non-null    int64         
 4   revenue_sgd  325 non-null    float64       
 5   staff_hours  325 non-null    float64       
 6   headcount    325 non-null    int64         
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 20.3+ KB


In [30]:
# 3. Add a `rev_per_staff_hour` column.
clean['rev_per_staff_hour'] = clean['revenue_sgd'] / clean['staff_hours']
clean.head()

,outlet_id,week_start,tickets,items,revenue_sgd,staff_hours,headcount,rev_per_staff_hour
0,OUT-01,2024-01-01,1486,2145,11785.71,473.0,14,24.916934
1,OUT-01,2024-01-08,1607,2293,12761.65,421.6,12,30.269568
2,OUT-01,2024-01-15,1469,2104,11830.20,459.7,14,25.734610
3,OUT-01,2024-01-22,1556,2214,12165.29,446.1,13,27.270321
4,OUT-01,2024-01-29,1372,1985,11076.22,441.9,13,25.064992


In [31]:
# 4. Find the **10 worst weeks** by `rev_per_staff_hour` across the whole chain. Which outlet dominates that list, 
# and when do those weeks cluster?
worst_weeks = clean.nsmallest(10, 'rev_per_staff_hour')
worst_weeks.sort_values(by="week_start")
worst_weeks["outlet_id"].value_counts()

,outlet_id,week_start,tickets,items,revenue_sgd,staff_hours,headcount,rev_per_staff_hour
203,OUT-03,2024-11-25,969,1404,7664.01,437.1,13,17.533768
205,OUT-03,2024-12-09,972,1413,7743.65,399.6,12,19.378504
206,OUT-03,2024-12-16,944,1344,7714.06,409.6,12,18.833154
212,OUT-03,2025-01-27,957,1404,7771.50,429.9,13,18.077460
223,OUT-03,2025-04-14,915,1332,7644.26,403.9,12,18.926120
324,OUT-05,2025-05-26,390,564,3248.18,182.9,5,17.759322
229,OUT-03,2025-05-26,893,1305,7203.80,389.2,11,18.509250
230,OUT-03,2025-06-02,885,1302,7257.87,407.3,12,17.819470
232,OUT-03,2025-06-16,990,1446,7897.27,455.9,13,17.322373
233,OUT-03,2025-06-23,910,1330,7264.20,378.9,11,19.171813


OUT-03    9
OUT-05    1
Name: outlet_id, dtype: int64

OUT-03 Marina Bay dominated. A competitor opened next door on 4 November 2024. The figures fell right after.  
There is hardly any changes in staff numbers.

### Challenge 3: "Who is actually hitting target?" (Reshape + aggregate)

In [32]:
# 1. `melt` `targets_wide` into long format.
targets = targets_wide.melt(
    id_vars="outlet_id",        # keep this as a column
    var_name="month",           # the old column headers land here
    value_name="target_sgd",    # the numbers land here
)
targets.head()

,outlet_id,month,target_sgd
0,OUT-01,2024-01,51800.0
1,OUT-02,2024-01,45900.0
2,OUT-03,2024-01,46000.0
3,OUT-04,2024-01,32000.0
4,OUT-01,2024-02,52000.0


In [33]:
# 2. Build monthly actual revenue per outlet, and join the targets on outlet and month.
# only keep YYYY-MM format for month
m = (
    sales
    .groupby("outlet_id")
    .resample("MS", on="date")          # no label="left" needed
    .sum(numeric_only=True)
    .reset_index()
    .assign(month=lambda df: df["date"].dt.strftime("%Y-%m"))
    .drop(columns="date")
)
m.head()

# using an inner join here because we only want to keep months that have both actuals and targets
perf = m.merge(targets, on=["outlet_id", "month"], how="inner")
perf.head()

,outlet_id,tickets,items,revenue_sgd,month
0,OUT-01,6794,9743,53744.04,2024-01
1,OUT-01,6243,9068,49725.46,2024-02
2,OUT-01,6596,9708,51932.42,2024-03
3,OUT-01,6740,9889,53412.10,2024-04
4,OUT-01,6814,9919,54228.57,2024-05


,outlet_id,tickets,items,revenue_sgd,month,target_sgd
0,OUT-01,6794,9743,53744.04,2024-01,51800.0
1,OUT-01,6243,9068,49725.46,2024-02,52000.0
2,OUT-01,6596,9708,51932.42,2024-03,52100.0
3,OUT-01,6740,9889,53412.10,2024-04,52300.0
4,OUT-01,6814,9919,54228.57,2024-05,52500.0


In [34]:
# 3. Add a boolean `hit_target` column.
perf['hit_target'] = perf['revenue_sgd'] >= perf['target_sgd']
perf.head()
perf.tail()

,outlet_id,tickets,items,revenue_sgd,month,target_sgd,hit_target
0,OUT-01,6794,9743,53744.04,2024-01,51800.0,True
1,OUT-01,6243,9068,49725.46,2024-02,52000.0,False
2,OUT-01,6596,9708,51932.42,2024-03,52100.0,False
3,OUT-01,6740,9889,53412.10,2024-04,52300.0,True
4,OUT-01,6814,9919,54228.57,2024-05,52500.0,True


,outlet_id,tickets,items,revenue_sgd,month,target_sgd,hit_target
67,OUT-04,4468,6423,38455.80,2025-02,33400.0,True
68,OUT-04,5357,7789,45970.30,2025-03,33500.0,True
69,OUT-04,4891,7121,41891.75,2025-04,33600.0,True
70,OUT-04,5133,7501,44123.43,2025-05,33700.0,True
71,OUT-04,5219,7548,44699.83,2025-06,33800.0,True


In [35]:
# 4. Produce a table with **one row per outlet** showing months hit, months missed, and hit rate as a percentage. 
# Sort it worst-first.
summary = perf.groupby("outlet_id", observed=True).agg(
    months=("hit_target", "size"),
    months_hit=("hit_target", "sum"),
)
summary["months_missed"] = summary["months"] - summary["months_hit"]
summary["hit_rate"] = (summary["months_hit"] / summary["months"] *  100).round(1)
summary = summary.sort_values("hit_rate", ascending=True)
summary

,months,months_hit,months_missed,hit_rate
outlet_id,,,,
OUT-01,18,5,13,27.8
OUT-03,18,5,13,27.8
OUT-02,18,7,11,38.9
OUT-04,18,16,2,88.9


5. One sentence: does the hit rate tell the same story as the revenue trend from class? If not, why not?
> **Answer:** The hit rate is dependent on the hit target. In this case OUT-04 has a target that is far lower than the other branches, hence the hit rate is very much higher. We may need to review this.   
OUT-01 and OUT-03 has the same hit rate of 27.8%. However, the cannibalisation for OUT-03 Marina Bay is not evident here at all.

### 🏆 Stretch Challenge
The competitor opened on **4 November 2024**. Did it hurt every day of the week equally, or is the
damage concentrated?

Build a table with `weekday` down the side and two columns — average daily revenue for the 8 weeks
*before* and the 8 weeks *after* — plus the percentage change. Then say what you would do with the
answer.

In [36]:
marina = sales[sales["outlet_id"] == "OUT-03"].copy()
marina["weekday"] = marina["date"].dt.day_name()

# One row per day first, or you will be averaging dayparts instead of days.
daily = marina.groupby(["date", "weekday"])["revenue_sgd"].sum().reset_index()

# find the 8 weeks before and after 4 Nov 2024
start_date = pd.to_datetime("2024-11-04") - pd.Timedelta(weeks=8)
end_date = pd.to_datetime("2024-11-04") + pd.Timedelta(weeks=8)

before = daily[(daily["date"] >= start_date) & (daily["date"] < "2024-11-04")]
after = daily[(daily["date"] >= "2024-11-04") & (daily["date"] < end_date)]

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

cmp = pd.DataFrame({
    "before": before.groupby("weekday")["revenue_sgd"].mean().round(0),
    "after": after.groupby("weekday")["revenue_sgd"].mean().round(0),
}).loc[order]
cmp["change_pct"] = ((cmp["after"] / cmp["before"] - 1) * 100).round(1)

print(cmp)

           before   after  change_pct
weekday                              
Monday     1685.0  1382.0       -18.0
Tuesday    1864.0  1389.0       -25.5
Wednesday  1754.0  1306.0       -25.5
Thursday   1744.0  1410.0       -19.2
Friday     1642.0  1261.0       -23.2
Saturday    863.0   709.0       -17.8
Sunday      638.0   505.0       -20.8


> **Answer:** There is general drop in revenue after the competitor opened. They are spaced throughout the week, with no clear indication if it's affecting weekdays or weekends more.  
Furthermore the sample size is only for 8 weeks before & after, so noise is expected for such a small sample.  

## 💬 Reflection
In class, the inner join silently removed $61,310 from the chain total. You would not have noticed
if the notebook had not printed both numbers side by side. What will you *actually do* differently
— a specific habit, in code — to catch that in your own work? Be concrete enough that you could
write it as a checklist item.  
> **Answer:** After every .merge() / .join(), compare len() before and after. If any rows disappeared, print the unmatched keys with an anti-join before proceeding:

In [37]:
# anti-join: what's in left but NOT in merged?
missing = left[~left["key"].isin(right["key"])]
print(missing)

NameError: name 'left' is not defined